# 03 — X-IIoTID Data Preparation + Excel Copies

**Goal:** Keep all 820,834 records in the verified frozen train/validation/test split.

**Files:** Each CSV.GZ and its Excel copy are stored together in `data/splits/x_iiotid/`.

**Rule:** Excel is for reading only. Models use the compressed CSV files, with labels excluded from inputs.


In [ ]:
from pathlib import Path
import hashlib, json, gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
DATA = ROOT / 'data/processed/x_iiotid/x_iiotid_core19_selected_v3.csv.gz'
MANIFEST = ROOT / 'data/splits/x_iiotid/x_iiotid_core19_signature_split_manifest_v1.csv'
AUDIT1 = ROOT / 'reports/audits/x_iiotid_v3_dataset_analysis.json'
AUDIT2 = ROOT / 'reports/audits/x_iiotid_v3_feature_quality.json'
OUT_DIR = ROOT / 'data/splits/x_iiotid'
REPORT = ROOT / 'reports/audits/x_iiotid_v3_split_report.json'
FIGURE = ROOT / 'reports/figures/x_iiotid_v3_split_class_balance.png'
PARTS = ('train', 'validation', 'test')
EXPECTED = {'train': 574585, 'validation': 123125, 'test': 123124}
CHUNK = 30000
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown'}

for file in (DATA, MANIFEST, AUDIT1, AUDIT2):
    if not file.is_file():
        raise FileNotFoundError(f'Missing input: {file}. Keep the frozen split; do not resplit automatically.')
a1 = json.loads(AUDIT1.read_text(encoding='utf-8'))
a2 = json.loads(AUDIT2.read_text(encoding='utf-8'))
FEATURES = a2['selected_features']
FIELDS = ['csv_row_index_0based'] + FEATURES + ['class1','class2','class3']
assert len(FEATURES) == 19 and a2['rows'] == a1['rows'] == 820834
assert a2['source_sha256_from_notebook_01'] == a1['source_sha256']
assert a2['exported_columns'] == FIELDS
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)
FIGURE.parent.mkdir(parents=True, exist_ok=True)
print('Selected dataset:', DATA.name)
print('Using existing frozen manifest:', MANIFEST.name)

## 1. Verify the frozen split

Check row IDs, target labels and saved partition sizes. The split must match the previous experiments.

In [ ]:
manifest = pd.read_csv(MANIFEST, dtype={'csv_row_index_0based':'int64',
                                             'partition':'str', 'class3':'str'}, low_memory=False)
for name in ('csv_row_index_0based', 'partition', 'class3'):
    if name not in manifest.columns:
        raise ValueError(f'Manifest missing column: {name}')
if len(manifest) != a1['rows']:
    raise ValueError('Frozen manifest row count does not match the reduced dataset.')
if not np.array_equal(manifest['csv_row_index_0based'].to_numpy(), np.arange(len(manifest))):
    raise ValueError('Manifest original row IDs must be complete and in original CSV order.')
if not manifest.partition.isin(PARTS).all() or not manifest.class3.isin(['Normal','Attack']).all():
    raise ValueError('Unexpected partition or label in manifest.')
counts = {p: int((manifest.partition == p).sum()) for p in PARTS}
if counts != EXPECTED:
    raise ValueError(f'Frozen split counts changed: {counts} != {EXPECTED}')
print('Frozen partition sizes:', counts)

## 2. Verify feature-signature separation

Check that identical normalized Core19 feature patterns do not occur in different partitions. If they do, **stop and investigate** rather than silently creating a new split.

In [ ]:
# The same text normalization used in Notebook 02 (diagnostic only).
from hashlib import blake2b
signature_partition = {}
overlap_groups = set()
partition_counts = {p: {'Normal': 0, 'Attack': 0} for p in PARTS}
seen_rows = 0

for chunk in pd.read_csv(DATA, dtype=str, keep_default_na=False, chunksize=CHUNK, low_memory=False):
    if chunk.columns.tolist() != FIELDS:
        raise ValueError('Reduced dataset schema differs from Notebook 02.')
    size = len(chunk)
    target = manifest.iloc[seen_rows:seen_rows+size]
    if len(target) != size:
        raise ValueError('Reduced dataset contains more records than manifest.')
    record_ids = pd.to_numeric(chunk['csv_row_index_0based'], errors='raise').to_numpy()
    if not np.array_equal(record_ids, target['csv_row_index_0based'].to_numpy()):
        raise ValueError(f'Reduced dataset row IDs do not match near row {seen_rows}.')
    classes = chunk['class3'].str.strip().to_numpy()
    if not np.array_equal(classes, target['class3'].to_numpy()):
        raise ValueError(f'Reduced dataset labels do not match near row {seen_rows}.')
    parts = target['partition'].to_numpy()
    for part in PARTS:
        rows_here = classes[parts == part]
        partition_counts[part]['Normal'] += int((rows_here == 'Normal').sum())
        partition_counts[part]['Attack'] += int((rows_here == 'Attack').sum())
    norm = chunk[FEATURES].apply(lambda col: col.str.strip().mask(
        col.str.strip().str.casefold().isin(MISSING), '<MISSING>'))
    for values, part in zip(norm.itertuples(index=False, name=None), parts):
        digest = blake2b(json.dumps(values, ensure_ascii=False,
                                   separators=(',', ':')).encode('utf-8'), digest_size=16).digest()
        previous = signature_partition.setdefault(digest, part)
        if previous != part:
            overlap_groups.add(digest)
    seen_rows += size

if seen_rows != len(manifest):
    raise ValueError(f'Reduced dataset row count mismatch: {seen_rows} != {len(manifest)}')
if overlap_groups:
    raise ValueError(f'{len(overlap_groups):,} normalized Core19 feature patterns cross frozen partitions. '
                     'Stop: inspect split compatibility before training. Do not resplit silently.')
print('Verified records:', f'{seen_rows:,}')
print('Overlapping normalized feature groups across partitions: 0')
display(pd.DataFrame([{'partition': p, 'normal': partition_counts[p]['Normal'],
                       'attack': partition_counts[p]['Attack'], 'total': counts[p]} for p in PARTS]))

## 3. Export or reuse the frozen split files

Keep all records. Existing V3 split files are reused and verified instead of overwritten.


In [ ]:
outputs = {p: OUT_DIR / f'x_iiotid_core19_{p}_v3.csv.gz' for p in PARTS}
temporary = {p: Path(str(outputs[p]) + '.tmp') for p in PARTS}
for path in temporary.values():
    if path.exists():
        raise FileExistsError(f'Incomplete split file found: {path}. Inspect before rerunning.')

# Reuse complete V3 files rather than rewriting verified partitions.
existing = {p: outputs[p].is_file() for p in PARTS}
if any(existing.values()) and not all(existing.values()):
    raise FileNotFoundError(f'Only some V3 splits exist: {existing}. Review before replacing them.')

if all(existing.values()):
    written = EXPECTED.copy()
    print('Reusing existing V3 split files; checking every row in the next cell.')
else:
    written = {p: 0 for p in PARTS}
    header_needed = {p: True for p in PARTS}
    import contextlib
    try:
        with contextlib.ExitStack() as stack:
            streams = {p: stack.enter_context(gzip.open(temporary[p], 'wt',
                              encoding='utf-8', newline='')) for p in PARTS}
            offset = 0
            for chunk in pd.read_csv(DATA, dtype=str, keep_default_na=False,
                                     chunksize=CHUNK, low_memory=False):
                parts = manifest.partition.iloc[offset:offset+len(chunk)].to_numpy()
                for p in PARTS:
                    sub = chunk.loc[parts == p]
                    if not sub.empty:
                        sub.to_csv(streams[p], header=header_needed[p], index=False)
                        header_needed[p] = False
                        written[p] += len(sub)
                offset += len(chunk)
        if written != EXPECTED:
            raise ValueError(f'Export sizes incorrect: {written}')
        for p in PARTS:
            temporary[p].replace(outputs[p])
    except BaseException:
        for path in temporary.values():
            if path.exists():
                path.unlink()
        raise
for p in PARTS:
    print(p, f'{written[p]:,} records ->', outputs[p].name)


In [ ]:
# Verify the saved files really contain every record and all 23 expected columns.
verified = {}
for part in PARTS:
    total = 0
    expected_ids = manifest.loc[manifest['partition'] == part, 'csv_row_index_0based'].to_numpy()
    start = 0
    for piece in pd.read_csv(outputs[part], dtype=str, keep_default_na=False,
                             chunksize=CHUNK, low_memory=False):
        if piece.columns.tolist() != FIELDS:
            raise ValueError(f'{part}: exported columns do not match Notebook 02.')
        ids = pd.to_numeric(piece['csv_row_index_0based'], errors='raise').to_numpy()
        if not np.array_equal(ids, expected_ids[start:start+len(piece)]):
            raise ValueError(f'{part}: exported row IDs differ from frozen manifest.')
        total += len(piece)
        start += len(piece)
    if total != EXPECTED[part]:
        raise ValueError(f'{part}: expected {EXPECTED[part]:,} rows, found {total:,}.')
    verified[part] = total
if sum(verified.values()) != a1['rows']:
    raise ValueError('Records were lost or added during export.')
print(f"All rows preserved: {sum(verified.values()):,}/{a1['rows']:,} (100%).")
print('Split totals:', verified)


## 4. Save the split report and graph

Save the counts and verify that the held-out test partition was not scored.


In [ ]:
summary = pd.DataFrame([{'partition': p, 'Normal': partition_counts[p]['Normal'],
                                  'Attack': partition_counts[p]['Attack']} for p in PARTS]).set_index('partition')
ax = summary.plot(kind='bar', stacked=True, figsize=(8,4.5), rot=0)
ax.set_title('X-IIoTID — Frozen Train / Validation / Test Split')
ax.set_ylabel('Records')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE, dpi=180, bbox_inches='tight')
plt.show()
plt.close()

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

report = {
    'dataset': 'X-IIoTID', 'feature_set': 'Core19', 'features': FEATURES,
    'target': 'class3', 'extra_labels': ['class1','class2'],
    'source_reduced_dataset': str(DATA.relative_to(ROOT)),
    'source_reduced_dataset_sha256': file_sha256(DATA),
    'original_source_sha256': a1['source_sha256'],
    'frozen_manifest': str(MANIFEST.relative_to(ROOT)),
    'manifest_sha256': file_sha256(MANIFEST),
    'counts': written, 'verified_export_counts': verified,
    'total_rows_preserved': sum(verified.values()), 'rows_removed': 0,
    'class_counts': partition_counts,
    'normalized_signature_overlap_groups': len(overlap_groups),
    'export_files': {p: str(outputs[p].relative_to(ROOT)) for p in PARTS},
    'preprocessing_fitted': False, 'test_scored': False,
    'note': 'Existing frozen split reused; no new split or model fitting. Session independence unverified.'
}
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved report:', REPORT)
print('Next: 04_model_training.ipynb — fit preprocessing on training data only.')

## 5. Excel copies alongside the split files

Create or reuse one Excel workbook per split. The CSV.GZ files remain the data used for training and evaluation.


In [ ]:
import subprocess

EXCEL_EXPORT = True  # Set False only if Microsoft Excel desktop is not installed.
EXCEL_SCRIPT = ROOT / 'scripts/export_x_iiotid_splits_excel.ps1'
if EXCEL_EXPORT:
    if not EXCEL_SCRIPT.is_file():
        raise FileNotFoundError(f'Copy the Excel export script to: {EXCEL_SCRIPT}')
    result = subprocess.run(
        ['powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass',
         '-File', str(EXCEL_SCRIPT), '-ProjectRoot', str(ROOT)],
        check=True
    )
    EXCEL_DIR = OUT_DIR
    for part in PARTS:
        excel_file = EXCEL_DIR / f'x_iiotid_core19_{part}_v3.xlsx'
        if not excel_file.is_file():
            raise FileNotFoundError(f'Excel export missing: {excel_file}')
        print(f'{part}: {EXPECTED[part]:,} records in {excel_file.name}')
    print('Excel files are for review only. Train and evaluate using CSV.GZ files.')
else:
    print('Excel export skipped. Existing split CSV.GZ files are unchanged.')
